In [17]:
import pandas as pd
import numpy as np

import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.utils import limpiar_df

pd.set_option('display.max_columns', None)

In [16]:
ruta = '../data/raw/fbref_big5'

In [12]:
misc = pd.read_parquet('../data/raw/fbref_big5/2021-2022/big5_2021-2022_misc.parquet')

In [13]:
misc.head(3)

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
                  Rk             Player             Nation                Pos   
0                  1         Max Aarons            eng ENG                 DF   
1                  2   Yunis Abdelhamid             ma MAR                 DF   
2                  3  Salis Abdul Samed             gh GHA                 MF   

  Unnamed: 4_level_0  Unnamed: 5_level_0 Unnamed: 6_level_0  \
               Squad                Comp                Age   
0       Norwich City  eng Premier League                 21   
1              Reims          fr Ligue 1                 33   
2      Clermont Foot          fr Ligue 1                 21   

  Unnamed: 7_level_0 Unnamed: 8_level_0 Performance                         \
                Born                90s        CrdY CrdR 2CrdY Fls Fld Off   
0               2000               32.0           8    0     0  26  52   1   
1               1987               33.1           5    1     0  38  25   0   
2               2000               27.4          12    3     2  43  37   0   

                              Unnamed: 21_level_0  
  Crs Int TklW PKwon PKcon OG             Matches  
0  50  26   45   NaN   NaN  1             Matches  
1   4  67   28   NaN   NaN  0             Matches  
2  14  40   26   NaN   NaN  0             Matches

In [14]:
misc = limpiar_df(misc)

In [15]:
misc.head(3)

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,90s,Performance_CrdY,Performance_CrdR,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21,2000,32.0,8,0,0,26,52,1,50,26,45,NaN,NaN,1
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33,1987,33.1,5,1,0,38,25,0,4,67,28,NaN,NaN,0
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21,2000,27.4,12,3,2,43,37,0,14,40,26,NaN,NaN,0


In [22]:
RAW = Path(ruta)

In [23]:
for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name          # '2021-2022'
    for archivo in sorted(carpeta.glob("*.parquet")):
        print(temporada, archivo.stem)

2021-2022 big5_2021-2022_misc
2021-2022 big5_2021-2022_playingtime
2021-2022 big5_2021-2022_shooting
2021-2022 big5_2021-2022_stats
2022-2023 big5_2022-2023_misc
2022-2023 big5_2022-2023_playingtime
2022-2023 big5_2022-2023_shooting
2022-2023 big5_2022-2023_stats
2023-2024 big5_2023-2024_misc
2023-2024 big5_2023-2024_playingtime
2023-2024 big5_2023-2024_shooting
2023-2024 big5_2023-2024_stats
2024-2025 big5_2024-2025_misc
2024-2025 big5_2024-2025_playingtime
2024-2025 big5_2024-2025_shooting
2024-2025 big5_2024-2025_stats
2025-2026 big5_2025-2026_misc
2025-2026 big5_2025-2026_playingtime
2025-2026 big5_2025-2026_shooting
2025-2026 big5_2025-2026_stats


In [49]:
por_temporada = []
CLAVE = ['Player', 'Squad', 'Comp', 'Born']

for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name

    # 1. cargar las cuatro tablas de esta temporada
    tablas = {}
    for archivo in carpeta.glob("*.parquet"):
        tipo = archivo.stem.split("_")[-1]      # 'stats', 'misc', ...
        tablas[tipo] = limpiar_df(pd.read_parquet(archivo))
    # 2. unirlas entre sí  -> df_temporada
    # ...  aquí va tu lógica de merge
    

    base = tablas['stats'].copy()

    for tipo in ['shooting', 'misc', 'playingtime']:
        otra = tablas[tipo]
        nuevas = [c for c in otra.columns if c not in base.columns]
        base = base.merge(
            otra[CLAVE + nuevas],
            on=CLAVE,
            how='left',
            validate='one_to_one'
        )

    df_temporada = base

    df_temporada["Season"] = temporada
    por_temporada.append(df_temporada)

# 3. apilar todas las temporadas
completo = pd.concat(por_temporada, ignore_index=True)

In [64]:
completo['Performance_Gls'] = pd.to_numeric(completo['Performance_Gls'], errors='coerce')

In [65]:
completo.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21,2000,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,NaN,NaN,1,85,84.2,32,89,29,2,22,2,0.53,18,69,-51,-1.59,+0.08,2021-2022
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33,1987,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,NaN,NaN,0,88,87.2,34,88,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21,2000,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,NaN,NaN,0,79,72.0,29,82,19,2,38,1,1.10,26,43,-17,-0.62,+0.69,2021-2022
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28,1993,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,NaN,NaN,0,87,86.4,34,87,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21,2000,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,NaN,NaN,0,45,1.3,1,45,0,0,NaN,0,1.00,0,0,0,0.00,+0.93,2021-2022


In [66]:
completo.sort_values(by = 'Performance_Gls', ascending = False).head(10)

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
4050,1129,Erling Haaland,NOR,"FW,MF",Manchester City,Premier League,22,2000,35,33,2769,30.8,36,8,44,29,7,7,5,0,1.17,0.26,1.43,0.94,1.20,30.8,36,123,60,48.8,4.00,1.95,0.29,0.60,7,7,0,31,24,8,13,3,2,NaN,NaN,0,79,81.0,33,83,21,2,22,1,2.37,84,26,+58,+1.89,+1.47,2022-2023
7103,1293,Harry Kane,ENG,FW,Bayern Munich,Bundesliga,30,1993,32,32,2839,31.5,36,8,44,31,5,5,2,0,1.14,0.25,1.39,0.98,1.24,31.5,36,146,67,45.9,4.63,2.12,0.25,0.54,5,5,0,10,39,13,30,8,9,NaN,NaN,0,89,92.8,32,89,27,0,NaN,0,2.16,85,39,+46,+1.46,+0.24,2023-2024
12836,1320,Harry Kane,ENG,"FW,MF",Bayern Munich,Bundesliga,32,1993,31,25,2377,26.4,36,5,41,26,10,11,1,0,1.36,0.19,1.55,0.98,1.17,26.4,36,119,67,56.3,4.51,2.54,0.30,0.54,10,11,0,25,32,20,12,8,13,NaN,NaN,0,77,77.7,25,86,17,6,38,1,2.58,98,26,+72,+2.73,+0.88,2025-2026
1504,1505,Robert Lewandowski,POL,FW,Bayern Munich,Bundesliga,32,1988,34,34,2946,32.7,35,3,38,30,5,5,2,0,1.07,0.09,1.16,0.92,1.01,32.7,35,161,81,50.3,4.92,2.47,0.22,0.43,5,5,0,22,37,20,28,6,2,NaN,NaN,0,87,96.3,34,87,28,0,NaN,0,2.26,92,36,+56,+1.71,-1.45,2021-2022
10353,1691,Kylian Mbappé,FRA,FW,Real Madrid,La Liga,25,1998,34,34,2907,32.3,31,3,34,24,7,9,3,1,0.96,0.09,1.05,0.74,0.84,32.3,31,161,85,52.8,4.98,2.63,0.19,0.36,7,9,0,25,40,44,34,1,5,NaN,NaN,0,86,85.0,34,86,22,0,NaN,0,2.24,67,32,+35,+1.08,+0.21,2024-2025
4270,1349,Harry Kane,ENG,FW,Tottenham,Premier League,29,1993,38,38,3405,37.8,30,3,33,25,5,6,6,0,0.79,0.08,0.87,0.66,0.74,37.8,30,132,64,48.5,3.49,1.69,0.23,0.47,5,6,0,32,56,15,37,2,12,NaN,NaN,0,90,99.6,38,90,35,0,NaN,0,1.58,70,62,+8,+0.21,+6.21,2022-2023
10967,2305,Mohamed Salah,EGY,MF,Liverpool,Premier League,32,1992,38,38,3371,37.5,29,18,47,20,9,9,1,0,0.77,0.48,1.25,0.53,1.01,37.5,29,130,61,46.9,3.47,1.63,0.22,0.48,9,9,0,25,39,18,79,9,11,NaN,NaN,0,89,98.6,38,89,29,0,NaN,0,2.21,85,40,+45,+1.20,+1.20,2024-2025
4645,1724,Kylian Mbappé,FRA,FW,PSG,Ligue 1,23,1998,34,32,2818,31.3,29,5,34,26,3,4,6,0,0.93,0.16,1.09,0.83,0.99,31.3,29,150,79,52.7,4.79,2.52,0.19,0.37,3,4,0,19,40,33,48,6,4,NaN,NaN,0,83,82.4,32,86,26,2,34,0,2.24,73,33,+40,+1.28,-0.07,2022-2023
6870,1060,Serhou Guirassy,GUI,FW,Stuttgart,Bundesliga,27,1996,28,25,2208,24.5,28,3,31,24,4,4,0,0,1.14,0.12,1.26,0.98,1.10,24.5,28,91,45,49.5,3.71,1.83,0.31,0.62,4,4,0,18,55,17,1,6,3,NaN,NaN,0,79,72.2,25,85,15,3,28,0,2.39,58,24,+34,+1.39,+0.86,2023-2024
1712,1713,Kylian Mbappé,FRA,FW,PSG,Ligue 1,22,1998,35,34,3023,33.6,28,17,45,24,4,4,10,0,0.83,0.51,1.34,0.71,1.22,33.6,28,147,68,46.3,4.38,2.02,0.19,0.41,4,4,0,29,53,39,68,4,10,NaN,NaN,0,86,88.4,34,88,24,1,21,0,2.34,86,30,+56,+1.67,+2.12,2021-2022


In [56]:
completo[completo['Player'].str.contains('Lionel Messi')]

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
1769,1770,Lionel Messi,ARG,"FW,MF",PSG,Ligue 1,34,1987,26,24,2153,23.9,6,14,20,6,0,0,0,0,0.25,0.59,0.84,0.25,0.84,23.9,6,89,29,32.6,3.72,1.21,0.07,0.21,0,0,0,6,35,10,33,4,3,NaN,NaN,0,83,63.0,24,88,22,2,27,0,2.12,54,23,+31,+1.30,-0.34,2021-2022
4694,1773,Lionel Messi,ARG,"FW,MF",PSG,Ligue 1,35,1987,32,32,2837,31.5,16,16,32,16,0,0,0,0,0.51,0.51,1.02,0.51,1.02,31.5,16,129,57,44.2,4.09,1.81,0.12,0.28,0,0,0,8,36,6,92,2,18,NaN,NaN,0,89,83.0,32,89,27,0,NaN,0,2.34,77,31,+46,+1.46,+1.00,2022-2023


In [63]:
completo[['Performance_Gls']].dtypes

Performance_Gls    str
dtype: object